# Coupling-interval refinement with the Julia-backed full worm

The original modWorm PPC execution is a **partitioned** neural/body method. Internally, CVODE_BDF and Tsit5 choose their own substeps, but muscle forces and PPC are updated only at sample boundaries. We test the effect of reducing the **sample/coupling interval** \(H\), not merely changing the solver tolerances. This is a numerical sensitivity experiment, not a guaranteed converged reference.

In [ ]:
from pathlib import Path
import os, numpy as np, matplotlib.pyplot as plt
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'modWorm'/'__init__.py').exists()),None)
if ROOT is None:raise RuntimeError('Start notebook from inside a modWorm checkout')
os.chdir(ROOT)
from modWorm import predefined_classes_nv as nv, predefined_classes_mb as mb, utils
from modWorm.julia_multiscale import run_modulated_julia, JuliaRunSettings
GAP,SYN=utils.construct_connectome_Cook()
MAP=np.load('modWorm/muscle_maps/muscle_map_adjust.npy')
# Defined physical exposure: constant zero imposed neural stimulation.
I0=np.zeros(279)
T=0.02
SETTINGS=JuliaRunSettings()


In [ ]:
def run_at_interval(h):
    count=int(round(T/h))+1
    assert abs((count-1)*h-T)<1e-10
    np.random.seed(42)
    n=nv.CelegansWorm_NervousSystem_PPC_Julia(GAP,SYN)
    b=mb.CelegansWorm_MuscleBody_PPC_Julia(MAP)
    n.timescale=h
    # Preserve physical PPC delay and onset: these durations divide every h here.
    n.fdb_init_Int=int(round(1.18/h))
    n.fdb_delay_Int=int(round(0.6/h))
    return run_modulated_julia(n,b,np.tile(I0,(count,1)),settings=SETTINGS)

step_sizes=[.01,.005,.0025]
runs={h:run_at_interval(h) for h in step_sizes}


In [ ]:
fine=runs[.0025]
for h in step_sizes[:-1]:
    co=runs[h]
    stride=int(round(h/.0025))
    assert np.allclose(co['time'],fine['time'][::stride])
    neural_error=np.max(np.abs(co['v_solution']-fine['v_solution'][::stride]))
    head_error=np.max(np.linalg.norm(np.column_stack((co['x_solution'][:,0],co['y_solution'][:,0]))-
                         np.column_stack((fine['x_solution'][::stride,0],fine['y_solution'][::stride,0])),axis=1))
    print('H:',h,'max voltage diff:',neural_error,'head xy diff:',head_error,
          'wall seconds:',co['elapsed_seconds'])
# A finite difference between two grids is NOT a verified global error bound.
fig,ax=plt.subplots(figsize=(7,4))
for h,res in runs.items():
    ax.plot(res['time'],res['x_solution'][:,0],label=f'H={h}')
ax.legend();ax.set(xlabel='simulation time (s)',ylabel='head x (model units)');plt.show()


The 0.02 s run **does not activate** the original 1.18 s PPC feedback. A scientifically meaningful delay test needs a longer run after the performance problem is resolved. If refinements fail or errors increase, stop; inspect the neural Jacobian, resting-threshold factorization, and solver diagnostics before changing the physical model.